LightGBM Approach

In [ ]:
# Run this in a cell first if you don't have it installed
!pip install lightgbm

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import time
import gc
import re
import pyarrow.parquet as pq

In [ ]:
# --- 1. CONFIGURATION ---
DRIVE_PATH = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
SPLIT_DIR = f"{DRIVE_PATH}/processed_data/final_foundation/splits"
METADATA_PATH = f"{DRIVE_PATH}/processed_data/items_metadata_final.parquet"

# --- 2. LAZY BIAS PRE-CALCULATION ---
print("🧮 Pre-calculating Collaborative Features (Lazy Load)...")
train_lazy = pl.scan_parquet(f"{SPLIT_DIR}/train.parquet")

user_stats = train_lazy.group_by("user_id").agg([
    pl.col("Rating").mean().alias("user_avg").cast(pl.Float32),
    pl.col("Rating").count().alias("user_count").cast(pl.Float32)
]).collect()

item_stats = train_lazy.group_by("item_id").agg([
    pl.col("Rating").mean().alias("item_avg").cast(pl.Float32),
    pl.col("Rating").count().alias("item_count").cast(pl.Float32)
]).collect()

global_mean = train_lazy.select(pl.count("Rating")).collect().item()

🧮 Pre-calculating Collaborative Features (Lazy Load)...


In [ ]:
# --- 3. METADATA SANITIZATION (The Error Fix) ---
print("🧹 Cleaning Feature Names for LightGBM...")
meta_df = pl.read_parquet(METADATA_PATH)

# Identify numeric columns
raw_meta_cols = [
    col for col in meta_df.columns
    if meta_df[col].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64]
    and col not in ['BGGId', 'item_id', 'Title', 'Description']
]

# Create a mapping: replace anything that isn't a letter or number with an underscore
rename_map = {col: re.sub(r'[^A-Za-z0-9_]', '_', col) for col in raw_meta_cols}

# Apply the clean names to our metadata dataframe
meta_df_clean = meta_df.select(["item_id"] + raw_meta_cols).rename(rename_map)

# Update our active feature list to the clean names
meta_cols = list(rename_map.values())
print(f"✅ Sanitized {len(meta_cols)} features.")

🧹 Cleaning Feature Names for LightGBM...
✅ Sanitized 416 features.


In [ ]:
# --- 4. THE OUT-OF-CORE BATCH GENERATOR (PyArrow Fix) ---
# Ensure our stats dataframes are fully loaded into memory (they are tiny)
# If you ran Section 2 earlier, these are already collected.
user_stats_mem = user_stats.collect() if isinstance(user_stats, pl.LazyFrame) else user_stats
item_stats_mem = item_stats.collect() if isinstance(item_stats, pl.LazyFrame) else item_stats
meta_df_mem = meta_df_clean.collect() if isinstance(meta_df_clean, pl.LazyFrame) else meta_df_clean

feature_names = ["user_avg", "user_count", "item_avg", "item_count"] + meta_cols

def process_in_batches(file_path, batch_size=500_000):
    """Uses PyArrow for true sequential streaming (Zero Memory Leaks)"""
    parquet_file = pq.ParquetFile(file_path)

    # iter_batches reads exactly the chunk we need and nothing else
    for record_batch in parquet_file.iter_batches(batch_size=batch_size):
        # Convert just this tiny chunk to a Polars DataFrame
        chunk_df = pl.from_arrow(record_batch)

        # Perform joins entirely in-memory on the chunk
        batch = (
            chunk_df
            .join(user_stats_mem, on="user_id", how="left")
            .join(item_stats_mem, on="item_id", how="left")
            .join(meta_df_mem, on="item_id", how="left")
            .with_columns([
                pl.col("user_avg").fill_null(global_mean),
                pl.col("item_avg").fill_null(global_mean),
                pl.col("user_count").fill_null(0.0),
                pl.col("item_count").fill_null(0.0)
            ])
            .fill_null(0.0)
        )

        X = batch.select(feature_names).to_numpy().astype(np.float32)
        y = batch["Rating"].to_numpy().astype(np.float32)

        yield X, y

        # Force cleanup
        del chunk_df, batch
        gc.collect()

In [ ]:
# --- 5. INCREMENTAL TRAINING ---
print(f"🚀 Training LightGBM Incrementally (500k rows per batch)...")
start_time = time.time()

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 63,
    'max_depth': 8,
    'feature_fraction': 0.8,
    'n_jobs': -1,
    'verbose': -1
}

model = None
batch_count = 0

for X_batch, y_batch in process_in_batches(f"{SPLIT_DIR}/train.parquet", batch_size=500_000):
    # free_raw_data=True tells LightGBM to dump the numpy array from RAM once it builds its internal tree structure
    train_data = lgb.Dataset(X_batch, label=y_batch, feature_name=feature_names, free_raw_data=True)

    model = lgb.train(
        params,
        train_data,
        num_boost_round=15,
        init_model=model,
        keep_training_booster=True
    )

    batch_count += 1
    print(f"  Processed {batch_count * 500_000:,} rows...")

    del X_batch, y_batch, train_data
    gc.collect()

print(f"✅ Training completed in {time.time() - start_time:.2f}s")

🚀 Training LightGBM Incrementally (500k rows per batch)...
  Processed 500,000 rows...
  Processed 1,000,000 rows...
  Processed 1,500,000 rows...
  Processed 2,000,000 rows...
  Processed 2,500,000 rows...
  Processed 3,000,000 rows...
  Processed 3,500,000 rows...
  Processed 4,000,000 rows...
  Processed 4,500,000 rows...
  Processed 5,000,000 rows...
  Processed 5,500,000 rows...
  Processed 6,000,000 rows...
  Processed 6,500,000 rows...
  Processed 7,000,000 rows...
  Processed 7,500,000 rows...
  Processed 8,000,000 rows...
  Processed 8,500,000 rows...
  Processed 9,000,000 rows...
  Processed 9,500,000 rows...
  Processed 10,000,000 rows...
  Processed 10,500,000 rows...
  Processed 11,000,000 rows...
  Processed 11,500,000 rows...
  Processed 12,000,000 rows...
  Processed 12,500,000 rows...
  Processed 13,000,000 rows...
  Processed 13,500,000 rows...
  Processed 14,000,000 rows...
  Processed 14,500,000 rows...
  Processed 15,000,000 rows...
✅ Training completed in 637.36s


In [ ]:
import lightgbm as lgb

# --- 1. SAVE THE MODEL ---
# We save it as a .txt file, which is LightGBM's native format
MODEL_PATH = f"{DRIVE_PATH}/processed_data/lgbm_batch_hybrid.txt"

model.save_model(MODEL_PATH)
print(f"✅ Success! LightGBM model saved to:\n{MODEL_PATH}")


# --- 2. HOW TO LOAD IT LATER (Example) ---
"""
When you come back to Colab tomorrow, you don't need to run the training cell.
Just run your setup/imports, and then load the model like this:

print("Loading saved model...")
loaded_model = lgb.Booster(model_file=MODEL_PATH)

# You can then use loaded_model.predict() exactly as before!
"""

✅ Success! LightGBM model saved to:
/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/lgbm_batch_hybrid.txt


'\nWhen you come back to Colab tomorrow, you don\'t need to run the training cell. \nJust run your setup/imports, and then load the model like this:\n\nprint("Loading saved model...")\nloaded_model = lgb.Booster(model_file=MODEL_PATH)\n\n# You can then use loaded_model.predict() exactly as before!\n'

In [ ]:

# --- 2. HOW TO LOAD IT LATER (Example) ---
"""
When you come back to Colab tomorrow, you don't need to run the training cell.
Just run your setup/imports, and then load the model like this:
"""
print("Loading saved model...")
MODEL_PATH = f"{DRIVE_PATH}/processed_data/lgbm_batch_hybrid.txt"
model = lgb.Booster(model_file=MODEL_PATH)

# You can then use loaded_model.predict() exactly as before!


Loading saved model...


In [ ]:
# --- 6. BATCH EVALUATION ---
print("\n📊 Evaluating on Test Set...")
total_se = 0.0
total_n = 0

# FIXED: Removed the ', _' since PyArrow only yields X and y
for X_test_batch, y_test_batch in process_in_batches(f"{SPLIT_DIR}/test.parquet", batch_size=500_000):
    preds = np.clip(model.predict(X_test_batch), 1, 10)
    total_se += np.sum((y_test_batch - preds) ** 2)
    total_n += len(y_test_batch)

    del X_test_batch, y_test_batch
    gc.collect()

rmse = np.sqrt(total_se / total_n)

print("-" * 45)
print(f"  LIGHTGBM BATCH HYBRID FINAL RMSE: {rmse:.4f}")
print("-" * 45)


📊 Evaluating on Test Set...
---------------------------------------------
  LIGHTGBM BATCH HYBRID FINAL RMSE: 1.2379
---------------------------------------------


In [ ]:
from sklearn.metrics import ndcg_score

# --- 6.5 ADVANCED METRICS (NDCG & COVERAGE) ---
print("\n🚀 Calculating NDCG@10 & Coverage (Sampling 500 Users)...")
start_metrics = time.time()

# 1. Grab 500 Random Users from the Test Set (Lazy Load for memory safety)
test_lazy = pl.scan_parquet(f"{SPLIT_DIR}/test.parquet")
unique_test_users = test_lazy.select("user_id").unique().collect()["user_id"].to_list()
sample_users = np.random.choice(unique_test_users, size=min(500, len(unique_test_users)), replace=False)

# 2. Calculate NDCG@10
ndcg_test_df = test_lazy.filter(pl.col("user_id").is_in(sample_users)).collect()

ndcg_features = (
    ndcg_test_df
    .join(user_stats_mem, on="user_id", how="left")
    .join(item_stats_mem, on="item_id", how="left")
    .join(meta_df_mem, on="item_id", how="left")
    .with_columns([
        pl.col("user_avg").fill_null(global_mean),
        pl.col("item_avg").fill_null(global_mean),
        pl.col("user_count").fill_null(0.0),
        pl.col("item_count").fill_null(0.0)
    ])
    .fill_null(0.0)
)

X_ndcg = ndcg_features.select(feature_names).to_numpy().astype(np.float32)
ndcg_features = ndcg_features.with_columns(pl.Series("predicted", model.predict(X_ndcg)))

ndcg_scores = []
for _, group in ndcg_features.group_by("user_id"):
    actual = [group["Rating"].to_numpy()]
    predicted = [group["predicted"].to_numpy()]
    if len(actual[0]) > 1:  # We need at least 2 items to rank them
        ndcg_scores.append(ndcg_score(actual, predicted, k=10))

final_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0

# 3. Calculate Catalog Coverage
# Build a base matrix of ALL 22,000 items
all_items = meta_df_mem.join(item_stats_mem, on="item_id", how="left").with_columns([
    pl.col("item_avg").fill_null(global_mean),
    pl.col("item_count").fill_null(0.0)
]).fill_null(0.0)

item_matrix_base = all_items.select(["item_avg", "item_count"] + meta_cols).to_numpy().astype(np.float32)
all_item_ids = all_items["item_id"].to_numpy()
total_catalog_size = len(all_item_ids)

recommended_set = set()
user_stats_dict = {
    row["user_id"]: (row["user_avg"], row["user_count"])
    for row in user_stats_mem.filter(pl.col("user_id").is_in(sample_users)).to_dicts()
}

# Generate Top 10 recommendations for our 500 users
for user in sample_users:
    u_avg, u_count = user_stats_dict.get(user, (global_mean, 0.0))
    u_features = np.tile([u_avg, u_count], (total_catalog_size, 1)).astype(np.float32)

    X_cov = np.hstack([u_features, item_matrix_base])
    preds = model.predict(X_cov)

    top_10_idx = np.argsort(-preds)[:10]
    recommended_set.update(all_item_ids[top_10_idx])

    del X_cov, preds

final_coverage = len(recommended_set) / total_catalog_size

print("-" * 45)
print(f"  LIGHTGBM NDCG@10:  {final_ndcg:.4f}")
print(f"  LIGHTGBM COVERAGE: {final_coverage:.2%}")
print(f"  (Calculated in {time.time() - start_metrics:.2f}s)")
print("-" * 45)


🚀 Calculating NDCG@10 & Coverage (Sampling 500 Users)...
---------------------------------------------
  LIGHTGBM NDCG@10:  0.9721
  LIGHTGBM COVERAGE: 0.47%
  (Calculated in 292.46s)
---------------------------------------------


In [ ]:
# --- 7. FEATURE IMPORTANCE ---
print("\nTop 5 Most Important Features:")
importance = model.feature_importance(importance_type='gain')
top_indices = np.argsort(importance)[-5:][::-1]
for i in top_indices:
    print(f" - {feature_names[i]}")


Top 5 Most Important Features:
 - item_avg
 - user_avg
 - AvgRating
 - user_count
 - StdDev
